[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgcalvo/SP1306/blob/main/edps-parabolicas/MEF_calor1D.ipynb)

# Ecuación del calor en 1D con elementos P1

Traducción de `MEF_calor1D.m`.

$$u_t = u_{xx} \quad \text{en } (0,1)\times(0,T], \qquad u(0,t)=u(1,t)=0,
\qquad u(x,0)=u_0(x)$$

con elementos finitos P1 en el espacio, avanzando en el tiempo de dos maneras:
**Euler hacia atrás** (implícito) y **Euler hacia adelante** (explícito).

## Formulación débil

Multiplicando por $v$ que se anula en los extremos e integrando por partes solo
el término de segundo orden,

$$(u_t, v) + (u_x, v_x) = (f, v) \qquad \forall v$$

Al escribir $u_h(x,t)=\sum_j u_j(t)\,\varphi_j(x)$ aparecen las **dos** matrices
del ensamblaje: la de masa viene del término temporal y la de rigidez del
término espacial,

$$M\,\mathbf{u}'(t) + A\,\mathbf{u}(t) = \mathbf{b}, \qquad
M_{ij}=\int \varphi_i\varphi_j, \qquad A_{ij}=\int \varphi_i'\varphi_j'$$

Esta es la diferencia con el problema estacionario, donde $M$ no se usa.

## Los dos esquemas

Según dónde se evalúe el término $A\mathbf{u}$:

$$\text{hacia atrás:}\quad M\frac{\mathbf{u}^{n+1}-\mathbf{u}^n}{\Delta t} + A\mathbf{u}^{n+1} = \mathbf{b}
\quad\Longrightarrow\quad (M + \Delta t\,A)\,\mathbf{u}^{n+1} = M\mathbf{u}^n + \Delta t\,\mathbf{b}$$

$$\text{hacia adelante:}\quad M\frac{\mathbf{u}^{n+1}-\mathbf{u}^n}{\Delta t} + A\mathbf{u}^{n} = \mathbf{b}
\quad\Longrightarrow\quad M\,\mathbf{u}^{n+1} = (M - \Delta t\,A)\,\mathbf{u}^n + \Delta t\,\mathbf{b}$$

Un detalle que conviene no pasar por alto: en elementos finitos el método
"explícito" **también** pide resolver un sistema en cada paso, porque la matriz
de masa $M$ no es diagonal. Lo explícito es el tratamiento del operador
espacial, no el costo del paso. (La forma usual de volverlo genuinamente
explícito es *condensar* la masa, reemplazando $M$ por la diagonal de las sumas
de sus filas; no se hace aquí.)

En los dos casos la matriz que hay que invertir —$M+\Delta t\,A$ en uno, $M$ en
el otro— **no cambia** de un paso al otro, porque ni la malla ni $\Delta t$
dependen del tiempo. Se factoriza **una sola vez**, fuera del ciclo, y cada paso
es solo una sustitución. Es el mismo criterio que lleva a ensamblar de una sola
vez en vez de entrada por entrada.

## Estabilidad

Aquí está la diferencia de fondo. Si $\lambda$ son los valores propios de
$A\mathbf{v}=\lambda M\mathbf{v}$, el factor de amplificación de cada modo es

$$\text{hacia atrás:}\quad \frac{1}{1+\Delta t\,\lambda} < 1 \ \text{ siempre}
\qquad\qquad
\text{hacia adelante:}\quad 1 - \Delta t\,\lambda, \quad |\cdot|\le 1 \ \text{ solo si } \
\Delta t \le \frac{2}{\lambda_{\max}}$$

El implícito es **incondicionalmente estable**: $\Delta t$ y $h$ se eligen por
precisión. El explícito no. Para P1 en una malla uniforme se puede calcular

$$\lambda_{\max} \longrightarrow \frac{12}{h^2}
\qquad\text{de donde}\qquad
\Delta t^* = \frac{2}{\lambda_{\max}} = \frac{h^2}{6}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu, eigsh

## Entradas

Con $u_0=\sin(\pi x)$ la solución exacta es $e^{-\pi^2 t}\sin(\pi x)$: la forma
se conserva y solo decae la amplitud.

In [ ]:
T   = 0.1                                          # tiempo final
u0f = lambda x: np.sin(np.pi*x)                    # dato inicial
f   = lambda x: np.zeros_like(x)                   # sin fuente: u_t = u_xx
uex = lambda x, t: np.exp(-np.pi**2*t)*np.sin(np.pi*x)

## La malla

`node` es el vector de coordenadas y `elem` la conectividad: la fila $e$ tiene
los dos índices de los extremos del elemento $e$. En NumPy los índices arrancan
en 0, así que los mismos números aparecen corridos en 1 respecto al `.m`.

In [ ]:
def malla1D(a, b, N):
    """Malla de N nodos en [a,b]: node es (N,) y elem es (N-1, 2)."""
    node = np.linspace(a, b, N)
    elem = np.column_stack([np.arange(N-1), np.arange(1, N)])
    return node, elem

## Ensamblaje

`ensamblar1D` es la versión en una dimensión de `assembling.m`: mismo formato de
tripletes y mismo ciclo sobre los pares de funciones de forma, con
$2\times 2=4$ entradas por elemento en vez de $3\times 3=9$.

Sobre un elemento de longitud $h$ las matrices locales son

$$A_{\text{loc}} = \frac{1}{h}\begin{pmatrix} 1 & -1 \\ -1 & 1\end{pmatrix}
\qquad\qquad
M_{\text{loc}} = \frac{h}{6}\begin{pmatrix} 2 & 1 \\ 1 & 2\end{pmatrix}$$

que en forma compacta, para el ciclo vectorizado, se escriben

```
sA = (2*(i==j) - 1)/h            sM = h*((i==j) + 1)/6
```

La segunda es la hermana en 1D de la fórmula `area*((i==j)+1)/12` del código en
2D.

In [ ]:
def ensamblar1D(node, elem, f):
    """Rigidez, masa y lado derecho P1 en 1D (version 1D de assembling.m)."""
    N = len(node)
    h = node[elem[:, 1]] - node[elem[:, 0]]        # longitud de cada elemento

    ii, jj, sA, sM = [], [], [], []
    for i in range(2):
        for j in range(2):
            ii.append(elem[:, i])
            jj.append(elem[:, j])
            sA.append((2*(i == j) - 1)/h)          # (1/h)[1 -1; -1 1]
            sM.append(h*((i == j) + 1)/6)          # (h/6)[2 1; 1 2]
    ii = np.concatenate(ii); jj = np.concatenate(jj)
    A = coo_matrix((np.concatenate(sA), (ii, jj)), shape=(N, N)).tocsr()
    M = coo_matrix((np.concatenate(sM), (ii, jj)), shape=(N, N)).tocsr()

    # lado derecho con la regla del punto medio, exacta para grado 1
    medio = (node[elem[:, 0]] + node[elem[:, 1]])/2
    bt = np.column_stack([f(medio)*h/2, f(medio)*h/2])
    b = np.bincount(elem.ravel(order='F'), weights=bt.ravel(order='F'), minlength=N)
    return A, M, b, h

## Los dos avanzadores en el tiempo

`splu` hace el papel de `decomposition` de MATLAB: factoriza la matriz una vez y
después cada paso es solo una sustitución. Las dos funciones son idénticas salvo
por *cuál* matriz se factoriza y *qué* se arma en el lado derecho.

In [ ]:
def resolver_BE(node, elem, dt, T, u0f, f):
    """Euler hacia atras:  (M + dt A) u^{n+1} = M u^n + dt b."""
    N = len(node)
    A, M, b, _ = ensamblar1D(node, elem, f)

    free = np.arange(1, N-1)              # Dirichlet homogeneo en ambos extremos
    u = u0f(node).astype(float)
    u[0] = u[-1] = 0.0
    if T == 0:
        return u

    nsteps = max(1, int(round(T/dt)))
    dt = T/nsteps                         # ajusta dt para caer exactamente en T

    # La matriz del sistema no depende del paso: se factoriza una sola vez.
    Mf = M[free][:, free].tocsr()
    lu = splu((Mf + dt*A[free][:, free]).tocsc())

    uf, bf = u[free], b[free]
    for n in range(nsteps):
        uf = lu.solve(Mf @ uf + dt*bf)
    u[free] = uf
    return u


def resolver_FE(node, elem, dt, T, u0f, f):
    """Euler hacia adelante:  M u^{n+1} = (M - dt A) u^n + dt b.

    Aunque el esquema sea explicito, con masa consistente hay que resolver con M
    en cada paso. M tampoco cambia, asi que se factoriza una sola vez.
    Es estable solo si dt <= dt_critico(node, elem)."""
    N = len(node)
    A, M, b, _ = ensamblar1D(node, elem, f)

    free = np.arange(1, N-1)
    u = u0f(node).astype(float)
    u[0] = u[-1] = 0.0
    if T == 0:
        return u

    nsteps = max(1, int(round(T/dt)))
    dt = T/nsteps

    Mf = M[free][:, free].tocsr()
    Af = A[free][:, free].tocsr()
    lu = splu(Mf.tocsc())

    uf, bf = u[free], b[free]
    for n in range(nsteps):
        uf = lu.solve(Mf @ uf - dt*(Af @ uf) + dt*bf)
    u[free] = uf
    return u

## Dos utilidades: el paso crítico y el error en $L^2$

In [ ]:
def dt_critico(node, elem):
    """Paso maximo que admite Euler hacia adelante: 2/lambda_max.

    lambda_max es el mayor valor propio de A v = lambda M v. En una malla
    uniforme tiende a 12/h^2, de donde dtc -> h^2/6."""
    N = len(node)
    A, M, _, _ = ensamblar1D(node, elem, lambda x: np.zeros_like(x))
    free = np.arange(1, N-1)
    lmax = eigsh(A[free][:, free].tocsc(), k=1, M=M[free][:, free].tocsc(),
                 which='LA', return_eigenvectors=False)[0]
    return 2/lmax


def error_L2(node, elem, u, uexT):
    """Norma L2 del error, con Gauss de 3 puntos en cada elemento.

    Exacta para grado 5, de sobra para comparar una funcion lineal a trozos
    contra una suave."""
    xi = np.array([-np.sqrt(3/5), 0.0, np.sqrt(3/5)])
    w  = np.array([5/9, 8/9, 5/9])

    x1, x2 = node[elem[:, 0]], node[elem[:, 1]]
    u1, u2 = u[elem[:, 0]], u[elem[:, 1]]
    h = x2 - x1

    e2 = 0.0
    for q in range(len(xi)):
        xq = (x1 + x2)/2 + h/2*xi[q]                     # punto de cuadratura
        uq = u1*(1 - xi[q])/2 + u2*(1 + xi[q])/2         # u_h es lineal aqui
        e2 += np.sum(w[q]*(uq - uexT(xq))**2 * h/2)
    return np.sqrt(e2)

## 1. Una corrida

In [ ]:
N  = 101                               # nodos
dt = T/100                             # paso de tiempo

node, elem = malla1D(0, 1, N)
uBE = resolver_BE(node, elem, dt, T, u0f, f)

print(f'N = {N} nodos, dt = {dt:.2e}, {round(T/dt)} pasos')
print(f'dt critico del explicito en esta malla: {dt_critico(node, elem):.3e}')
print('\nEuler hacia atras:')
print(f'  error max en los nodos, t = T: {np.linalg.norm(uBE - uex(node, T), np.inf):.4e}')
print(f'  error en L2,            t = T: {error_L2(node, elem, uBE, lambda x: uex(x, T)):.4e}')

## 2. Evolución en el tiempo

In [ ]:
instantes = [0, 0.005, 0.02, 0.05, 0.1]
colores = plt.cm.tab10(np.arange(len(instantes)))

fig, ax = plt.subplots(figsize=(7, 4.5))
for c, tk in zip(colores, instantes):
    uk = resolver_BE(node, elem, dt, tk, u0f, f)
    ax.plot(node, uk, '-', color=c, lw=1.6, label=f't = {tk:g}')
    ax.plot(node, uex(node, tk), '--k', lw=0.9)
ax.set_xlabel('x'); ax.set_ylabel('u'); ax.legend()
ax.set_title('Euler implicito (lineas) vs solucion exacta (punteado)')
plt.show()

## 3. Convergencia en el tiempo

Se fija una malla fina para que el error espacial no contamine la medición. Solo
el implícito: en esta malla el explícito necesitaría $\Delta t < 4\times10^{-8}$.

In [ ]:
nodeF, elemF = malla1D(0, 1, 2001)

print('   pasos      dt        err_max     orden')
ep = np.nan
for ns in [10, 20, 40, 80, 160]:
    dtk = T/ns
    uk = resolver_BE(nodeF, elemF, dtk, T, u0f, f)
    e  = np.linalg.norm(uk - uex(nodeF, T), np.inf)
    print(f'{ns:8d}  {dtk:.3e}  {e:.4e}  {np.log2(ep/e):7.2f}')
    ep = e
print('Euler hacia atras es de orden 1: el error se parte a la mitad.')

## 4. Convergencia en el espacio

Ahora al revés: $\Delta t$ muy pequeño para aislar el error de la discretización
espacial. Hacen falta muchísimos pasos justamente porque Euler hacia atrás es de
orden 1: para que su error quede por debajo del espacial hay que exagerar. Con
$\Delta t$ más grande los órdenes de abajo se inflan por encima de 2, que es la
señal de que las dos fuentes de error ya son comparables.

In [ ]:
dtf = T/200000
print('    N        h        err_max    orden      err_L2    orden')
emp = e2p = np.nan
for Nk in [11, 21, 41, 81]:
    nk, ek = malla1D(0, 1, Nk)
    uk = resolver_BE(nk, ek, dtf, T, u0f, f)
    em = np.linalg.norm(uk - uex(nk, T), np.inf)
    e2 = error_L2(nk, ek, uk, lambda x: uex(x, T))
    print(f'{Nk:5d}  {1/(Nk-1):.3e}  {em:.3e}  {np.log2(emp/em):7.2f}   '
          f'{e2:.3e}  {np.log2(e2p/e2):7.2f}')
    emp, e2p = em, e2

Para este dato inicial el error espacial se puede rastrear hasta su origen: el
valor propio discreto más pequeño cumple

$$\lambda_h = \pi^2 + \frac{\pi^4 h^2}{12} + O(h^4)$$

así que la solución numérica decae un poco **más rápido** de lo debido, y el
error en el instante $t$ vale aproximadamente $t\,(\lambda_h-\pi^2)e^{-\pi^2 t}$.
Con $h=0.1$ y $t=0.1$ eso da $3.0\times10^{-3}$, que es justo lo que mide la
tabla de arriba.

## 5. Implícito contra explícito: la restricción de estabilidad

Misma malla y mismo $\Delta t$ para los dos. El implícito funciona en toda la
columna; el explícito solo cuando $\Delta t$ queda por debajo de $\Delta t^*$, y
apenas se pasa no se degrada: **explota**.

In [ ]:
Ne = 41
nodeE, elemE = malla1D(0, 1, Ne)
dtc = dt_critico(nodeE, elemE)
he  = 1/(Ne-1)

print(f'===== IMPLICITO vs EXPLICITO  (N = {Ne}, h = {he:.4f}) =====')
print(f'dt* = 2/lambda_max = {dtc:.3e}   (h^2/6 = {he**2/6:.3e}, h^2/2 = {he**2/2:.3e})')
print('   pasos      dt      dt/dt*     err implic.   err explic.')
for ns in [100, 500, 1000, 2000]:
    dtk = T/ns
    eI = np.linalg.norm(resolver_BE(nodeE, elemE, dtk, T, u0f, f) - uex(nodeE, T), np.inf)
    eE = np.linalg.norm(resolver_FE(nodeE, elemE, dtk, T, u0f, f) - uex(nodeE, T), np.inf)
    print(f'{ns:8d}  {dtk:.3e}  {dtk/dtc:7.2f}   {eI:10.3e}    {eE:10.3e}')

El error del implícito **no baja monótonamente** en esa columna, y no es un error
del código: Euler hacia atrás decae de menos
($1/(1+\Delta t\lambda) > e^{-\Delta t\lambda}$) mientras que la discretización
espacial decae de más ($\lambda_h > \pi^2$). Los dos errores tienen signos
opuestos y se cancelan cerca de $\Delta t = 10^{-4}$.

Donde el explícito es estable, los dos dan prácticamente lo mismo: son el mismo
método de orden 1, y no hay razón para pagar la restricción sobre $\Delta t$
salvo que $M$ esté condensada.

## 6. Estabilidad incondicional del implícito

Un solo paso que cubre todo el intervalo de tiempo. El explícito explotaría sin
remedio; acá la solución sigue siendo razonable, apenas imprecisa, que es
exactamente lo que significa *estable pero inexacto*.

In [ ]:
print(f'===== IMPLICITO CON dt ENORME  (N = {N}) =====')
print('   pasos      dt       err_max    max|u|')
for ns in [1, 2, 5, 100]:
    uk = resolver_BE(node, elem, T/ns, T, u0f, f)
    print(f'{ns:8d}  {T/ns:.3e}  {np.linalg.norm(uk - uex(node, T), np.inf):.3e}  '
          f'{np.abs(uk).max():.4f}')

lim = dt_critico(node, elem)
print(f'En esta malla el explicito pediria dt <= {lim:.2e}, o sea al menos')
print(f'{int(np.ceil(T/lim))} pasos para llegar a T = {T:g}.')